# Matching doc2 installations to doc1 airfields

**Goal:** produce a new table containing every row from **doc2**, enriched with airfield details from **doc1** (`icao`, `geoloc`, `airfield_name`, `iata`, `country_name`, `cluster`).

Matching happens in two passes:

1. **Direct match** — doc2's `iata_code` vs doc1's `iata`.
2. **Fallback match** — for rows that didn't match, split doc2's `installation_name` on the `*` character and check each token against a reverse dictionary of every doc1 value.

Rows that match neither way are kept, with the doc1 columns left blank.

## 1. Configuration

Set the file names and column names here. This is the only cell most people need to edit.

In [ ]:
import pandas as pd

file1 = "file1.csv"             # doc1: the airfield reference table
file2 = "file2.csv"             # doc2: the installations to enrich

key1 = "iata"                  # key column in doc1
key2 = "iata_code"             # key column in doc2
name_col = "installation_name"  # doc2 column tokenized on '*' for the fallback match

# Columns pulled FROM doc1 onto matching doc2 rows:
doc1_cols = ["icao", "geoloc", "airfield_name", "iata", "country_name", "cluster"]

# Which doc1 columns feed the reverse dictionary used for token matching.
# NOTE: 'country_name' and 'cluster' are generic (e.g. USA, East) and can cause
# false matches. Drop them from this list to match only specific identifiers.
lookup_cols = ["icao", "geoloc", "airfield_name", "iata", "country_name", "cluster"]

## 2. Load the data

`encoding="latin-1"` is used because CSVs exported from Excel often aren't UTF-8. We also print the columns so you can confirm the names above are correct.

In [ ]:
df1 = pd.read_csv(file1, encoding="latin-1")
df2 = pd.read_csv(file2, encoding="latin-1")

print("doc1 columns:", list(df1.columns))
print("doc2 columns:", list(df2.columns))
print("doc2 total rows:", len(df2))

# Fail early if a configured column name is wrong.
for c in doc1_cols:
    assert c in df1.columns, f"'{c}' is not in doc1. Available: {list(df1.columns)}"
assert name_col in df2.columns, f"'{name_col}' is not in doc2. Available: {list(df2.columns)}"

df1.head()

## 3. Helper: normalize a value

Matching is case- and whitespace-insensitive. `norm_val` trims spaces and uppercases a value, returning `None` for blanks so empty cells never match each other.

In [ ]:
def norm_val(v):
    if pd.isna(v):
        return None
    s = str(v).strip().upper()
    return s if s else None

## 4. Build the doc1 lookups

Two dictionaries are built from doc1:

- **`key_lookup`** — normalized `iata` &rarr; that row's doc1 columns. Used for the direct match.
- **`value_dict`** — every value across `lookup_cols` &rarr; that row's doc1 columns. Used for token matching. First occurrence of a value wins.

In [ ]:
key_lookup = {}   # normalized iata -> dict of doc1_cols
value_dict = {}   # any normalized value from lookup_cols -> dict of doc1_cols

for _, row in df1.iterrows():
    info = {c: row[c] for c in doc1_cols}
    k = norm_val(row[key1])
    if k and k not in key_lookup:
        key_lookup[k] = info
    for c in lookup_cols:
        v = norm_val(row[c])
        if v and v not in value_dict:   # first occurrence wins
            value_dict[v] = info

print(f"key lookup: {len(key_lookup)} codes | reverse dict: {len(value_dict)} values")

## 5. Pass 1 — direct match on the key column

Start from a copy of doc2, add empty doc1 columns, then fill them wherever doc2's `iata_code` matches a doc1 `iata` code.

In [ ]:
matched = df2.copy()
for c in doc1_cols:
    matched[c] = None

direct = 0
for idx in matched.index:
    k = norm_val(matched.at[idx, key2])
    if k and k in key_lookup:
        for c, val in key_lookup[k].items():
            matched.at[idx, c] = val
        direct += 1

print(f"Direct iata_code matches: {direct}")

## 6. Pass 2 — fallback on `installation_name` tokens

For rows still unmatched, split `installation_name` on `*` and test each token against `value_dict`. The first token that hits fills in the doc1 columns.

In [ ]:
by_token = 0
for idx in matched.index:
    if matched.at[idx, doc1_cols[0]] is not None:   # already matched in pass 1
        continue
    raw = matched.at[idx, name_col]
    if pd.isna(raw):
        continue
    for tok in str(raw).split("*"):
        tok = tok.strip().upper()
        if tok and tok in value_dict:
            for c, val in value_dict[tok].items():
                matched.at[idx, c] = val
            by_token += 1
            break   # first matching token wins

print(f"Token (installation_name) matches: {by_token}")

## 7. Save and summarize

Write the enriched table to `matched.csv` and print a summary of how many rows matched each way.

In [ ]:
matched.to_csv("matched.csv", index=False)

n_match = direct + by_token
print(f"Wrote {len(matched)} rows to matched.csv")
print(f"  - direct iata_code matches: {direct}")
print(f"  - installation_name token matches: {by_token}")
print(f"  - total matched: {n_match}")
print(f"  - still no match: {len(matched) - n_match}")

matched.head()